# NDVI Change and Recovery After the 2017 Eagle Creek Fire (2014–2024)

The Eagle Creek Fire (2017) burned a large portion of the Columbia River Gorge in northern Oregon, affecting forest structure, recreation access, and viewsheds along the Interstate 84 corridor and the Historic Columbia River Highway. In this project, I use satellite-derived NDVI (Normalized Difference Vegetation Index) to quantify how vegetation greenness changed inside the fire perimeter and how it has recovered relative to nearby unburned areas.

**Research Question**

How did NDVI change inside the Eagle Creek Fire scar compared to surrounding unburned forests, and what does the NDVI trajectory suggest about vegetation recovery in the years following the fire?

**Objectives**

1. Delineate the Eagle Creek Fire burn area using an MTBS (Monitoring Trends in Burn Severity) fire-perimeter polygon.
2. Derive annual NDVI metrics for a multiyear period before and after the fire (e.g., 2014–2024).
3. Compare NDVI trends inside the burn area to a nearby unburned “control” region.
4. Visualize both:
   - A time series of NDVI inside vs. outside the burn.
   - A sequence of NDVI maps illustrating pre-fire conditions, immediate post-fire decline, and subsequent recovery.

## Background and Context

The Eagle Creek Fire started in early September 2017 in the Columbia River Gorge, eventually burning thousands of acres of mixed conifer and broadleaf forest. The Gorge is a high-visibility landscape that supports hiking, tourism, and transportation, so changes in vegetation cover are both ecologically and socially important.

NDVI (Normalized Difference Vegetation Index) is a widely used remote-sensing metric that approximates vegetation vigor and canopy density. NDVI is defined as:

\[
\text{NDVI} = \frac{NIR - Red}{NIR + Red}
\]

where *NIR* is reflectance in a near-infrared band and *Red* is reflectance in a red band. Higher NDVI values generally indicate more photosynthetically active vegetation.

Post-fire recovery studies often use NDVI (or related indices) to:

- Quantify immediate loss of canopy cover after fire.
- Track regrowth of shrubs, grasses, and trees over time.
- Compare burn severity classes or burned vs. unburned reference areas.

In this project, I focus on NDVI-based comparison of burned vs. unburned areas across multiple years, rather than detailed burn-severity classes, to keep the analysis reasonable.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr  
from shapely.geometry import Polygon

### maps and plots
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

### open street map
from osmnx import features as osm
import osmnx as ox

### file structure
import os
import pathlib

### earthpy
import earthpy
import earthpy.api.appeears as eaapp

import matplotlib.pyplot as plt

# Plot settings
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = False

xr.set_options(display_style="text")

In [ ]:
### set up project and folder for data
project = earthpy.Project("EagleCreek", dirname = 'eagle_creek_ndvi')

data_dir = Path("data")
fire_perimeter_path = data_dir / "or4563012190420170902_20160812_20180818_burn_bndy.shp"

In [ ]:
### Get Polygon for area of interest
hatfield_gdf = ox.geocode_to_gdf(
    'Mark O. Hatfield Wilderness')

### make a quick plot
hatfield_gdf.plot()

In [ ]:
# Years to analyze
pre_fire_years = list(range(2014, 2017))   # 2014, 2015, 2016
fire_year = 2017                           # fire start year
post_fire_years = list(range(2018, 2025))  # 2018–2024

all_years = pre_fire_years + [fire_year] + post_fire_years

all_years

In [ ]:
### initialize AppeearsDownloader for MODIS NDVI data
# ensure polygon is in geographic (lat/lon) before passing to AppEEARS
gdf_ll = fire_gdf.to_crs("EPSG:4326")

### set parameters
ndvi_downloader = eaapp.AppeearsDownloader(

    ### give your download a name
    download_key = "eagle_ndvi",

    ### tell it to put the data in your project that you already defined
    project = project,

    ### specify the MODIS product you want
    product = 'MOD13Q1.061',
    layer = '_250m_16_days_NDVI',

    ### choose a start date and end data
    start_date = "08-01",
    end_date = "08-30",

    ### recurring means you want those dates over multiple years
    recurring = True,

    ### specify the range of years you want
    year_range = [2014, 2024],

    ### specify the polygon you want to get NDVI data for
    polygon = gdf_11
)

In [ ]:
### download the prepared download -- this can take a while!
ndvi_downloader.download_files(cache=True)

In [ ]:
### get a sorted list of NDVI file paths
ndvi_paths = sorted(list(project.project_dir.rglob('*NDVI*.tif')))

ndvi_paths

In [ ]:
# Load the fire-perimeter polygon
fire_gdf = gpd.read_file(fire_perimeter_path)

print(fire_gdf.head())
print("\nCRS:", fire_gdf.crs)

if fire_gdf.crs is None or fire_gdf.crs.to_epsg() == 4326:
    projected_crs = "EPSG:5070"
    fire_gdf = fire_gdf.to_crs(projected_crs)
    print("\nReprojected fire perimeter to:", projected_crs)

# The burned area polygon(s)
burn_geom = fire_gdf.unary_union  # dissolve into a single polygon if there are multiple parts

In [ ]:
# Create a "ring" around the fire to act as an unburned comparison area.
# Idea: buffer the fire outward, then subtract the burned area to get an outside ring.

# Buffer distance in CRS units (meters if projected)
buffer_distance = 3000  # 3 km, can adjust

outer_buffer = burn_geom.buffer(buffer_distance)
outside_ring = outer_buffer.difference(burn_geom)

outside_gdf = gpd.GeoDataFrame(
    {"region": ["outside_burn"], "geometry": [outside_ring]},
    crs=fire_gdf.crs,
)

inside_gdf = gpd.GeoDataFrame(
    {"region": ["inside_burn"], "geometry": [burn_geom]},
    crs=fire_gdf.crs,
)

inside_gdf, outside_gdf

In [ ]:
def load_ndvi_for_year(year: int) -> xr.DataArray:
    """
    Load an annual (or seasonal) NDVI composite for a given year.

    This function assumes you have one NDVI raster per year named like:
        ndvi_YYYY.tif
    saved in `ndvi_dir`.

    Returns
    -------
    ndvi_da : xarray.DataArray
        NDVI raster for the given year with spatial coords and a CRS.
    """
    ndvi_path = ndvi_dir / f"ndvi_{year}.tif"
    if not ndvi_path.exists():
        raise FileNotFoundError(f"NDVI file not found: {ndvi_path}")

    ndvi_da = rioxarray.open_rasterio(ndvi_path).squeeze("band", drop=True)

    # Ensure NDVI is float and in the expected range [-1, 1]
    ndvi_da = ndvi_da.astype("float32")

    # If NDVI was stored as scaled integers (e.g., multiplied by 1000),
    # uncomment the line below:
    # ndvi_da = ndvi_da / 1000.0

    return ndvi_da

In [ ]:
def summarize_ndvi_for_year(year: int, inside_gdf: gpd.GeoDataFrame, outside_gdf: gpd.GeoDataFrame):
    """
    For a given year, load the NDVI composite, reproject to match the polygons
    if necessary, clip to inside and outside regions, and compute mean NDVI.

    Returns
    -------
    dict with keys: year, ndvi_inside_mean, ndvi_outside_mean
    """
    ndvi_da = load_ndvi_for_year(year)

    # Reproject NDVI to match vector CRS if needed
    if ndvi_da.rio.crs is None:
        raise ValueError("NDVI raster has no CRS; please ensure it is georeferenced.")

    if ndvi_da.rio.crs != inside_gdf.crs:
        ndvi_da = ndvi_da.rio.reproject(inside_gdf.crs)

    # Clip to inside burn
    ndvi_inside = ndvi_da.rio.clip(
        inside_gdf.geometry,
        inside_gdf.crs,
        drop=True,
        all_touched=True
    )

    # Clip to outside ring
    ndvi_outside = ndvi_da.rio.clip(
        outside_gdf.geometry,
        outside_gdf.crs,
        drop=True,
        all_touched=True
    )

    # Convert to NumPy arrays, mask NaNs
    inside_vals = ndvi_inside.values.ravel()
    outside_vals = ndvi_outside.values.ravel()

    inside_vals = inside_vals[~np.isnan(inside_vals)]
    outside_vals = outside_vals[~np.isnan(outside_vals)]

    result = {
        "year": year,
        "ndvi_inside_mean": float(inside_vals.mean()) if inside_vals.size > 0 else np.nan,
        "ndvi_outside_mean": float(outside_vals.mean()) if outside_vals.size > 0 else np.nan,
    }

    return result

In [ ]:
results = []

for yr in all_years:
    print(f"Processing NDVI for year {yr}...")
    try:
        res = summarize_ndvi_for_year(yr, inside_gdf, outside_gdf)
        results.append(res)
    except FileNotFoundError as e:
        print(f"  Skipping year {yr}: {e}")

ndvi_summary = pd.DataFrame(results).set_index("year").sort_index()
ndvi_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    ndvi_summary.index,
    ndvi_summary["ndvi_inside_mean"],
    marker="o",
    label="Inside burn"
)
ax.plot(
    ndvi_summary.index,
    ndvi_summary["ndvi_outside_mean"],
    marker="o",
    label="Outside (3 km ring)"
)

# Highlight fire year
if fire_year in ndvi_summary.index:
    ax.axvline(fire_year, linestyle="--", linewidth=1.5, label="Fire year")

ax.set_xlabel("Year")
ax.set_ylabel("Mean NDVI")
ax.set_title("Mean Annual NDVI Inside vs. Outside the Eagle Creek Fire Perimeter")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Choose years for mapping (adjust to match years you actually have)
pre_fire_map_year = 2014
post_fire_map_year = 2018
recovery_map_year = 2024

map_years = [pre_fire_map_year, post_fire_map_year, recovery_map_year]

ndvi_maps = {}
for yr in map_years:
    print(f"Loading NDVI for mapping: {yr}")
    ndvi_da = load_ndvi_for_year(yr)

    # Reproject and clip to a bounding box around the fire + buffer
    if ndvi_da.rio.crs != fire_gdf.crs:
        ndvi_da = ndvi_da.rio.reproject(fire_gdf.crs)

    # Build a combined bounding geometry (outer buffer) to restrict the map extent
    map_extent_geom = outer_buffer  # previously defined in AOI step

    ndvi_clipped = ndvi_da.rio.clip([map_extent_geom], fire_gdf.crs, drop=True, all_touched=True)
    ndvi_maps[yr] = ndvi_clipped

In [ ]:
def plot_ndvi_panel(ndvi_maps: dict, inside_gdf: gpd.GeoDataFrame, cmap: str = "YlGn"):
    """
    Plot NDVI maps for multiple years in a row of subplots.
    """
    n = len(ndvi_maps)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 6), sharey=True)

    if n == 1:
        axes = [axes]

    vmin = min([float(m.min()) for m in ndvi_maps.values()])
    vmax = max([float(m.max()) for m in ndvi_maps.values()])

    for ax, (yr, ndvi_da) in zip(axes, sorted(ndvi_maps.items())):
        im = ndvi_da.plot.imshow(
            ax=ax,
            add_colorbar=False,
            vmin=vmin,
            vmax=vmax,
            cmap=cmap
        )

        # Overlay fire perimeter
        inside_gdf.boundary.plot(ax=ax, edgecolor="red", linewidth=1)

        ax.set_title(f"NDVI {yr}")
        ax.set_xlabel("")
        ax.set_ylabel("")

    cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.04)
    cbar.set_label("NDVI")

    plt.suptitle("NDVI Before and After the 2017 Eagle Creek Fire", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()


plot_ndvi_panel(ndvi_maps, inside_gdf)